# Sympy code for FDM expansion of LBM diffusive equation up to Sixth-order

In [2]:
import sympy as sp
# -----------------------------------------------------------------------------
# 1) Symbols (parameters and step sizes)
# -----------------------------------------------------------------------------
dx, dt = sp.symbols('Delta_x Delta_t', positive=True, real=True)
s1, s2, w0, nu = sp.symbols('s_{1} s_{2} w_{0} \\nu', real=True)

def d(name, nx=0, nt=0):
    n = nx + nt
    if n == 1:
        num = rf"\partial{{{name}}}"
    else:
        num = rf"\partial^{{{n}}}{{{name}}}"
    den = ""
    if nx:
        den += rf"\partial{{x^{{{nx}}}}}" if nx > 1 else r"\partial{x}"
    if nt:
        den += rf"\partial{{t^{{{nt}}}}}" if nt > 1 else r"\partial{t}"
    return sp.Symbol(rf"\frac{{{num}}}{{{den}}}")
# -------------------1st-Order-Terms----------------------
dphit = d(r"\phi", nt=1); dphix   = d(r"\phi", nx=1)
# -------------------2nd-Order-Terms----------------------
dphix2 = d(r"\phi", nx=2); dphixt = d(r"\phi", nx=1, nt=1); dphit2 = d(r"\phi", nt=2)
# -------------------3rd-Order-Terms----------------------
dphix3 = d(r"\phi", nx=3); dphix2t = d(r"\phi", nx=2, nt=1); dphixt2 = d(r"\phi", nx=1, nt=2); dphit3  = d(r"\phi", nt=3)
# -------------------4th-Order-Terms----------------------
dphix4 = d(r"\phi", nx=4); dphix3t = d(r"\phi", nx=3, nt=1); dphix2t2 = d(r"\phi", nx=2, nt=2); dphixt3 = d(r"\phi", nx=1, nt=3); dphit4 = d(r"\phi", nt=4)
# -------------------5th-Order-Terms----------------------
dphix5 = d(r"\phi", nx=5); dphix4t = d(r"\phi", nx=4, nt=1); dphix3t2 = d(r"\phi", nx=3, nt=2); dphix2t3 = d(r"\phi", nx=2, nt=3); dphixt4 = d(r"\phi", nx=1, nt=4); dphit5 = d(r"\phi", nt=5)
# -------------------5th-Order-Terms----------------------
dphix6 = d(r"\phi", nx=6); dphix5t = d(r"\phi", nx=5, nt=1); dphix4t2 = d(r"\phi", nx=4, nt=2); dphix3t3 = d(r"\phi", nx=3, nt=3); dphix2t4 = d(r"\phi", nx=2, nt=4); dphixt5 = d(r"\phi", nx=1, nt=5); dphit6 = d(r"\phi", nt=6)

alpha1 = sp.simplify(1 - s1/2 - w0*s2/2)
alpha2 = sp.simplify((w0-1)*s2 + 1)
beta1  = sp.simplify(w0*s1*s2/2 - s1*s2/2 - w0*s2/2 + s1/2 +s2 -1)
beta2  = sp.simplify(-w0*s1*s2 + w0*s2 + s1 - 1)
gamma  = sp.simplify((s1-1)*(s2-1))
# -----------------------------------------------------------------------------
# 2) Continuous variables and field
# -----------------------------------------------------------------------------
x, t = sp.symbols('x t', real=True)
phi = sp.Function('phi')
# Shorthand for the base (continuous) field value at (x,t)
PHI = phi(x, t)

In [3]:
def taylor_phi(sx=0, st=0, order_x=6, order_t=6):
    expr = sp.S(0)
    # Keep all mixed terms up to the specified individual orders.
    for m in range(order_x + 1):
        for n in range(order_t + 1):
            # skip the (0,0) term? no, keep it.
            term = ( (sx*dx)**m * (st*dt)**n
                    / (sp.factorial(m)*sp.factorial(n))
                    * sp.diff(PHI, x, m, t, n) )
            expr += term
    return sp.expand(expr)

In [9]:
from IPython.display import display, Math
# phi_x^{t+1}, phi_x^{t}, phi_x^{t-1}, phi_x^{t-2}
phi_t_p1 = taylor_phi(sx=0,  st=+1, order_x=0, order_t=6)
display( Math(r"\phi_x^{t+1} = " + sp.latex(phi_t_p1)))
phi_t_0  = taylor_phi(sx=0,  st=0,  order_x=0, order_t=0)  # exactly phi(x,t)
display( Math(r"\phi_x^{t} = " + sp.latex(phi_t_0)))
phi_t_m1 = taylor_phi(sx=0,  st=-1, order_x=0, order_t=6)
display( Math(r"\phi_x^{t-1} = " + sp.latex(phi_t_m1)))
phi_t_m2 = taylor_phi(sx=0,  st=-2, order_x=0, order_t=6)
display( Math(r"\phi_x^{t-2} = " + sp.latex(phi_t_m2)))

# phi_{x±1}^{t}
phi_xm1_t0 = taylor_phi(sx=-1, st=0,  order_x=6, order_t=0)
display( Math(r"\phi_{x-1}^{t} = " + sp.latex(phi_xm1_t0)))
phi_xp1_t0 = taylor_phi(sx=+1, st=0,  order_x=6, order_t=0)
display( Math(r"\phi_{x+1}^{t} = " + sp.latex(phi_xp1_t0)))

# phi_{x±1}^{t-1}
phi_xm1_tm1 = taylor_phi(sx=-1, st=-1, order_x=6, order_t=6)
display( Math(r"\phi_{x-1}^{t-1} = " + sp.latex(phi_xm1_tm1)))
phi_xp1_tm1 = taylor_phi(sx=+1, st=-1, order_x=6, order_t=6)
display( Math(r"\phi_{x+1}^{t-1} = " + sp.latex(phi_xp1_tm1)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [10]:
# -----------------------------------------------------------------------------
# 5) Build the discrete equation (expanded) and move everything to one side
# -----------------------------------------------------------------------------
lhs = phi_t_p1

rhs = (alpha1*phi_xm1_t0 + alpha2*phi_t_0 + alpha1*phi_xp1_t0
       + beta1*phi_xm1_tm1 + beta2*phi_t_m1 + beta1*phi_xp1_tm1
       + gamma * phi_t_m2)

residual = sp.simplify(sp.expand(lhs - rhs))  # = 0 is the modified equation

In [11]:
max_dx=6
max_dt=3
max_total=6
expr = sp.expand(residual)
poly = sp.Poly(expr, dx, dt, domain='EX')  # keep symbolic coeffs
kept = sp.S(0)

for (px, pt), coeff in poly.terms():
    if (px <= max_dx) and (pt <= max_dt) and ((px + pt) <= max_total):
        kept += coeff * dx**px * dt**pt

In [13]:
term1 = sp.expand(kept).subs({
    # ------------------------------------------------6th order--------------------------------------------------------------------------------------------
    PHI.diff(x, 6): dphix6, PHI.diff(x, 5, t): dphix5t, PHI.diff(x, 4, t, 2): dphix4t2, PHI.diff(x, 3, t, 3): dphix3t3, PHI.diff(x, 2, t, 4): dphix2t4, PHI.diff(x, t, 5): dphixt5, PHI.diff(t, 6): dphit6,
    # ------------------------------------------------5th order--------------------------------------------------------------------------------------------
    PHI.diff(x, 5): dphix5, PHI.diff(x, 4, t): dphix4t, PHI.diff(x, 3, t, 2): dphix3t2, PHI.diff(x, 2, t, 3): dphix2t3, PHI.diff(x, t, 4): dphixt4, PHI.diff(t, 5): dphit5,
    # ------------------------------------------------4th order--------------------------------------------------------------------------------------------
    PHI.diff(x, 4):dphix4,PHI.diff(x, 3, t):dphix3t,PHI.diff(x, 2, t, 2):dphix2t2,PHI.diff(x, t, 3):dphixt3,PHI.diff(t, 4):dphit4,
    # ------------------------------------------------3rd order--------------------------------------------------------------------------------------------
    PHI.diff(x, 3):dphix3, PHI.diff(x, 2, t):dphix2t, PHI.diff(x, t, 2):dphixt2, PHI.diff(t, 3):dphit3,
    # ------------------------------------------------2nd order--------------------------------------------------------------------------------------------
    PHI.diff(x, 2):dphix2, PHI.diff(x, t):dphixt, PHI.diff(t, 2):dphit2,
    # ------------------------------------------------1st order--------------------------------------------------------------------------------------------
    PHI.diff(x):dphix, PHI.diff(t):dphit,
})
display(term1)

Delta_t**3*Delta_x**2*\frac{\partial^{5}{\phi}}{\partial{x^{2}}\partial{t^{3}}}*s_{1}*s_{2}*w_{0}/12 - Delta_t**3*Delta_x**2*\frac{\partial^{5}{\phi}}{\partial{x^{2}}\partial{t^{3}}}*s_{1}*s_{2}/12 + Delta_t**3*Delta_x**2*\frac{\partial^{5}{\phi}}{\partial{x^{2}}\partial{t^{3}}}*s_{1}/12 - Delta_t**3*Delta_x**2*\frac{\partial^{5}{\phi}}{\partial{x^{2}}\partial{t^{3}}}*s_{2}*w_{0}/12 + Delta_t**3*Delta_x**2*\frac{\partial^{5}{\phi}}{\partial{x^{2}}\partial{t^{3}}}*s_{2}/6 - Delta_t**3*Delta_x**2*\frac{\partial^{5}{\phi}}{\partial{x^{2}}\partial{t^{3}}}/6 + 7*Delta_t**3*\frac{\partial^{3}{\phi}}{\partial{t^{3}}}*s_{1}*s_{2}/6 - Delta_t**3*\frac{\partial^{3}{\phi}}{\partial{t^{3}}}*s_{1} - Delta_t**3*\frac{\partial^{3}{\phi}}{\partial{t^{3}}}*s_{2} + Delta_t**3*\frac{\partial^{3}{\phi}}{\partial{t^{3}}} - Delta_t**2*Delta_x**4*\frac{\partial^{6}{\phi}}{\partial{x^{4}}\partial{t^{2}}}*s_{1}*s_{2}*w_{0}/48 + Delta_t**2*Delta_x**4*\frac{\partial^{6}{\phi}}{\partial{x^{4}}\partial{t^{2}}}*s_{

In [17]:
term2 = sp.collect(sp.expand(-term1/(dt*s1*s2))+ dphit,[dphit,dphix,
                                                         dphix2*dx*dx/dt,dphixt,dphit2*dt,
                                                         dphix2t*dx*dx,dphixt2,dphix3,dphit3*dt**2,
                                                         dphix2t2*dt*dx**2,dphix4*dx**4/dt,
                                                         dphix4t*dx**4,dphix6*dx**6/dt,
                                                        ]).subs(dphix4t2,0).subs(dphix2t3,0)
display( Math(r"\frac{\partial\phi}{\partial t}= " + sp.latex(term2)))

<IPython.core.display.Math object>

In [21]:
nut=dx*dx*(w0/2-sp.Rational(1,2)-w0/s1+1/s1)/dt
term3 = sp.collect(sp.expand(term2.subs(nut,nu)
                       .subs(dphit2,nut*nut*dphix4)
                       .subs(dphit3,nut*nut*nut*dphix6)
                       .subs(dphix2t,nut*dphix4)
                        .subs(dphix2t2,nut*nut*dphix6)
                        .subs(dphix4t,nut*dphix6)
                            ), [dphix2*dx**2/dt,dphix4*dx**4/dt,dphix6*dx**4/dt])
display( Math(r"\frac{\partial\phi}{\partial t}= " + sp.latex(term3)))

<IPython.core.display.Math object>

## Fouth-Order Term

In [25]:
dphix4_term3=sp.simplify(sp.factor(term3.coeff(dphix4)).subs(dt*s1,(w0-1)*(2-s1)*dx**2/nu))
display(dphix4_term3)
# display( Math(r"\frac{\partial\phi}{\partial t}= " + sp.latex(term3.subs(term3.coeff(dphix4),dphix4_term3))))

Delta_x**2*\nu*(-3*s_{1}**2*s_{2}*w_{0} + 2*s_{1}**2*s_{2} + 6*s_{1}**2*w_{0} + 18*s_{1}*s_{2}*w_{0} - 12*s_{1}*s_{2} - 12*s_{1}*w_{0} - 12*s_{2}*w_{0} + 12*s_{2})/(24*s_{1}**2*s_{2})

In [34]:
dphix4_term3_2=sp.factor(dphix4_term3.coeff(dx**2*nu/(24*s1**2*s2)))
display(dphix4_term3_2)

-3*s_{1}**2*s_{2}*w_{0} + 2*s_{1}**2*s_{2} + 6*s_{1}**2*w_{0} + 18*s_{1}*s_{2}*w_{0} - 12*s_{1}*s_{2} - 12*s_{1}*w_{0} - 12*s_{2}*w_{0} + 12*s_{2}

In [35]:
print(sp.pycode(dphix4_term3_2))

-3*s_{1}**2*s_{2}*w_{0} + 2*s_{1}**2*s_{2} + 6*s_{1}**2*w_{0} + 18*s_{1}*s_{2}*w_{0} - 12*s_{1}*s_{2} - 12*s_{1}*w_{0} - 12*s_{2}*w_{0} + 12*s_{2}


In [92]:
eq_sol_s2 = (-3*s1**2*s2*w0 + 2*s1**2*s2 + 6*s1**2*w0 + 18*s1*s2*w0 - 12*s1*s2 - 12*s1*w0 - 12*s2*w0 + 12*s2)

sol_s2 = sp.solve(sp.Eq(eq_sol_s2, 0), s2)

display(sol_s2[0])
display(sol_s2[0].subs(w0,sp.Rational(2,3)))

6*s_{1}*w_{0}*(s_{1} - 2)/(3*s_{1}**2*w_{0} - 2*s_{1}**2 - 18*s_{1}*w_{0} + 12*s_{1} + 12*w_{0} - 12)

-s_{1}*(s_{1} - 2)

In [93]:
from sympy import latex
print(latex(sol_s2[0]))

\frac{6 s_{1} w_{0} \left(s_{1} - 2\right)}{3 s_{1}^{2} w_{0} - 2 s_{1}^{2} - 18 s_{1} w_{0} + 12 s_{1} + 12 w_{0} - 12}


In [97]:
# print(sp.pycode(latex(sol_s2[0])))

## Sixth-Order Term

In [46]:
dphix6_term3=sp.simplify(sp.factor(term3.coeff(dphix6)).subs(dt*s1,(w0-1)*(2-s1)*dx**2/nu))
display(dphix6_term3)

Delta_x**4*\nu*(60*s_{1}**3*s_{2}*w_{0}**2 - 105*s_{1}**3*s_{2}*w_{0} + 44*s_{1}**3*s_{2} - 90*s_{1}**3*w_{0}**2 + 135*s_{1}**3*w_{0} - 30*s_{1}**3 - 375*s_{1}**2*s_{2}*w_{0}**2 + 690*s_{1}**2*s_{2}*w_{0} - 300*s_{1}**2*s_{2} + 450*s_{1}**2*w_{0}**2 - 720*s_{1}**2*w_{0} + 240*s_{1}**2 + 690*s_{1}*s_{2}*w_{0}**2 - 1290*s_{1}*s_{2}*w_{0} + 600*s_{1}*s_{2} - 720*s_{1}*w_{0}**2 + 1260*s_{1}*w_{0} - 540*s_{1} - 360*s_{2}*w_{0}**2 + 720*s_{2}*w_{0} - 360*s_{2} + 360*w_{0}**2 - 720*w_{0} + 360)/(720*s_{1}**3*s_{2})

In [48]:
# dphix6_term3_2=sp.factor(dphix6_term3.coeff(dx**4*nu/(720*s1**3*s2)))
dphix6_term3_2=sp.factor(dphix6_term3.coeff(dx**4*nu))
display(dphix6_term3_2)

(60*s_{1}**3*s_{2}*w_{0}**2 - 105*s_{1}**3*s_{2}*w_{0} + 44*s_{1}**3*s_{2} - 90*s_{1}**3*w_{0}**2 + 135*s_{1}**3*w_{0} - 30*s_{1}**3 - 375*s_{1}**2*s_{2}*w_{0}**2 + 690*s_{1}**2*s_{2}*w_{0} - 300*s_{1}**2*s_{2} + 450*s_{1}**2*w_{0}**2 - 720*s_{1}**2*w_{0} + 240*s_{1}**2 + 690*s_{1}*s_{2}*w_{0}**2 - 1290*s_{1}*s_{2}*w_{0} + 600*s_{1}*s_{2} - 720*s_{1}*w_{0}**2 + 1260*s_{1}*w_{0} - 540*s_{1} - 360*s_{2}*w_{0}**2 + 720*s_{2}*w_{0} - 360*s_{2} + 360*w_{0}**2 - 720*w_{0} + 360)/(720*s_{1}**3*s_{2})

In [51]:
dphix6_term3_3 = sp.factor(dphix6_term3_2.subs(s2,sol_s2[0]))
dphix6_term3_3

(30*s_{1}**4*w_{0}**3 - 15*s_{1}**4*w_{0}**2 - 32*s_{1}**4*w_{0} + 20*s_{1}**4 + 60*s_{1}**3*w_{0}**3 - 420*s_{1}**3*w_{0}**2 + 600*s_{1}**3*w_{0} - 240*s_{1}**3 - 780*s_{1}**2*w_{0}**3 + 2580*s_{1}**2*w_{0}**2 - 2760*s_{1}**2*w_{0} + 960*s_{1}**2 + 1440*s_{1}*w_{0}**3 - 4320*s_{1}*w_{0}**2 + 4320*s_{1}*w_{0} - 1440*s_{1} - 720*w_{0}**3 + 2160*w_{0}**2 - 2160*w_{0} + 720)/(1440*s_{1}**4*w_{0})

In [78]:
eq_sol_w0 = (
    30*s1**4*w0**3 - 15*s1**4*w0**2 - 32*s1**4*w0 + 20*s1**4 + 60*s1**3*w0**3 - 420*s1**3*w0**2 + 
              600*s1**3*w0 - 240*s1**3 - 780*s1**2*w0**3 + 2580*s1**2*w0**2 - 2760*s1**2*w0 + 960*s1**2 + 1440*s1*w0**3 - 
              4320*s1*w0**2 + 4320*s1*w0 - 1440*s1 - 720*w0**3 + 2160*w0**2 - 2160*w0 + 720
            )


sol_w0 = sp.solve(sp.Eq(eq_sol_w0, 0), w0)

# display(sol_w0)
# display(sol_w0[0])
# display(sol_s2[0].subs(w0,sp.Rational(2,3)))

In [90]:
eq_sol_s1 = (
    30*s1**4*w0**3 - 15*s1**4*w0**2 - 32*s1**4*w0 + 20*s1**4 + 60*s1**3*w0**3 - 420*s1**3*w0**2 + 
              600*s1**3*w0 - 240*s1**3 - 780*s1**2*w0**3 + 2580*s1**2*w0**2 - 2760*s1**2*w0 + 960*s1**2 + 1440*s1*w0**3 - 
              4320*s1*w0**2 + 4320*s1*w0 - 1440*s1 - 720*w0**3 + 2160*w0**2 - 2160*w0 + 720
            ).subs(w0,sp.Rational(4,5))


sol_s1 = sp.solve(sp.Eq(eq_sol_s1, 0), s1)

display(sol_s1[0])
display(sol_s1[0].evalf())

-3 + sqrt(15)

0.872983346207417

In [59]:
display(sol_s1[0].subs(w0,sp.Rational(2,3)))
# display(sp.simplify(sol_s1[0].subs(w0,2/3)))

5/2 - sqrt(-5/3 + 80/(9*(475/54 + 5*sqrt(15)*I/6)**(1/3)) + 2*(475/54 + 5*sqrt(15)*I/6)**(1/3))/2 - sqrt(-10/3 - 2*(475/54 + 5*sqrt(15)*I/6)**(1/3) + 30/sqrt(-5/3 + 80/(9*(475/54 + 5*sqrt(15)*I/6)**(1/3)) + 2*(475/54 + 5*sqrt(15)*I/6)**(1/3)) - 80/(9*(475/54 + 5*sqrt(15)*I/6)**(1/3)))/2

In [73]:
value = sol_s1[0].subs(w0, sp.Rational(2, 6)).evalf()
display(value)

1.32188505494938 + 0.160475778168922*I

In [61]:
# print(sp.pycode(dphix6_term3_3))

In [15]:
# from sympy import latex
# print(latex(sp.simplify(kept.subs(dt**2*dx**2,0))))

In [67]:
1/1.20498996794432

0.8298824277400191